# Vorticity and velocity fields
Plot vorticity $\omega$ and the spectrally reconstructed velocity components $u$ and $v$ as separate PDF figures. Select one frame, several frames, or a pointwise frame average.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'ns2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from ns2d_plotting import (discover_vorticity, domain_lengths, frame_time,
    read_csv, read_parameters, read_vorticity, repository_root, save_figure,
    select_frames, use_plot_style, velocity_from_vorticity)

## Configuration

In [ ]:
ROOT = repository_root()  # Repository root; normally no change is needed.
DATA_DIRECTORY = ROOT / 'data'  # Directory containing vorticity_XXXXXXXX.dat files.
PARAMETER_FILE = ROOT / 'output/resolved_parameters.txt'  # Supplies the domain size.
DIAGNOSTICS_FILE = ROOT / 'output/diagnostics.csv'  # Supplies time labels when available.
VORTICITY_FIGURE = ROOT / 'figures/vorticity.pdf'
U_FIGURE = ROOT / 'figures/velocity_u.pdf'
V_FIGURE = ROOT / 'figures/velocity_v.pdf'

# 'single': one panel and exactly one selected frame.
# 'multiple': one panel for every selected frame.
# 'average': one panel averaged over all selected frames.
MODE = 'single'

# Explicit frame numbers. Negative indices count from the end, so [-1]
# selects the newest frame. Set to None to use START/STOP/STRIDE.
FRAMES = [-1]
FRAME_START = None
FRAME_STOP = None
FRAME_STRIDE = 1

PANEL_COLUMNS = 3
INTERPOLATION = 'bilinear'  # Use 'nearest' to display raw grid cells.
USE_TEX = True
FONT_SIZE = 15

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
files = discover_vorticity(DATA_DIRECTORY)
frames = select_frames(files, FRAMES, start=FRAME_START, stop=FRAME_STOP,
                       stride=FRAME_STRIDE)
if MODE == 'single' and len(frames) != 1:
    raise ValueError('single mode requires exactly one selected frame')
if MODE not in {'single', 'multiple', 'average'}:
    raise ValueError("MODE must be 'single', 'multiple', or 'average'")

parameters = read_parameters(PARAMETER_FILE) if PARAMETER_FILE.exists() else {}
lx, ly = domain_lengths(parameters)
extent = (0.0, lx, 0.0, ly)
diagnostics = read_csv(DIAGNOSTICS_FILE) if DIAGNOSTICS_FILE.exists() else None

records = []
for frame in frames:
    omega = read_vorticity(files[frame])
    u, v = velocity_from_vorticity(omega, lx, ly)
    label = f'frame {frame}'
    if diagnostics is not None and np.any(diagnostics['frame'] == frame):
        label += rf', $t={frame_time(diagnostics, frame):.4g}$'
    records.append((label, omega, u, v))

if MODE == 'average':
    records = [(f'average of {len(records)} frames',
                np.mean([record[1] for record in records], axis=0),
                np.mean([record[2] for record in records], axis=0),
                np.mean([record[3] for record in records], axis=0))]

omega_limit = max(max(np.max(np.abs(record[1])) for record in records),
                  np.finfo(float).eps)
velocity_limit = max(max(np.max(np.abs(record[index]))
                         for record in records for index in (2, 3)),
                     np.finfo(float).eps)
print(f'Selected frames: {frames}')

In [ ]:
def plot_quantity(field_index, title, limit, destination):
    count = len(records)
    columns = min(PANEL_COLUMNS, count)
    rows = int(np.ceil(count / columns))
    fig, axes = plt.subplots(rows, columns,
                             figsize=(5.0 * columns, 4.3 * rows),
                             squeeze=False)
    flat_axes = list(axes.flat)
    for axis, record in zip(flat_axes, records):
        image = axis.imshow(record[field_index], origin='lower', extent=extent,
                            interpolation=INTERPOLATION, cmap='RdBu_r',
                            vmin=-limit, vmax=limit)
        axis.set_title(title + '\n' + record[0])
        axis.set_xlabel(r'$x$')
        axis.set_ylabel(r'$y$')
        axis.set_aspect('equal')
    for axis in flat_axes[count:]:
        axis.set_visible(False)
    fig.colorbar(image, ax=flat_axes[:count], shrink=0.85, pad=0.03)
    saved = save_figure(fig, destination)
    print(f'Wrote {saved}')
    return fig

## Vorticity

In [ ]:
fig = plot_quantity(1, r'vorticity $\omega$', omega_limit, VORTICITY_FIGURE)
plt.show()

## Horizontal velocity

In [ ]:
fig = plot_quantity(2, r'velocity $u$', velocity_limit, U_FIGURE)
plt.show()

## Vertical velocity

In [ ]:
fig = plot_quantity(3, r'velocity $v$', velocity_limit, V_FIGURE)
plt.show()